In [ ]:
import pandas as pd
import numpy as np
import spacy
import string
import nltk
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, log_loss

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Profession-AI/progetti-ml/refs/heads/main/Modello%20per%20l'identificazione%20della%20lingua%20dei%20testi%20di%20un%20museo/museo_descrizioni.csv")

In [ ]:
df  #stampo il dataframe per esplorarlo

,Testo,Codice Lingua
0,Statua in marmo di un imperatore romano del II...,it
1,Anfora greca con decorazioni a figure nere,it
2,Dipinto rinascimentale raffigurante la Madonna...,it
3,Elmo corinzio in bronzo del VI secolo a.C.,it
4,Manoscritto medievale con miniature dorate,it
...,...,...
289,Wikinger-Anhänger mit Darstellung von Thor,de
290,Päpstliches Bleisiegel mit dem Bild des Heilig...,de
291,Leder-Gürtel mit langobardischer Bronzeschließe,de
292,Renaissance-Kupferbecken mit pflanzlichen Must...,de


In [ ]:
df["Codice Lingua"].value_counts()

,count
Codice Lingua,
it,98
en,98
de,98


In [ ]:
#Data cleaning
nltk.download('stopwords') #scarica le stopwords di NLTK
nlp = spacy.load('en_core_web_sm') #carica il modello spaCy (es. inglese)
stopwords_all = set(stopwords.words('english')) | set(stopwords.words('italian')) | set(stopwords.words('german')) #unisce stopwords delle tre lingue

#funzione di pulizia testo
def text_cleaner(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)  #rimuove punteggiatura
    text = re.sub(r"\d+", "", text)  #rimuove cifre

    doc = nlp(text)

    #lemmatizzazione + rimozione stopwords multilingua
    cleaned = " ".join(
        token.lemma_ for token in doc
        if token.lemma_ not in stopwords_all and token.lemma_ not in string.punctuation and not token.is_space)
    cleaned = re.sub(' +', ' ', cleaned)  #rimuove spazi multipli
    return cleaned.strip()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
corpus = df["Testo"].apply(text_cleaner) #applico la funzione alla colonna del df contenente i testi

In [ ]:
corpus #stampo i testi puliti

,Testo
0,statua marmo imperatore romano ii secolo
1,anfora greca decorazioni figure nere
2,dipinto rinascimentale raffigurante madonna ba...
3,elmo corinzio bronzo secolo
4,manoscritto medievale miniature dorate
...,...
289,wikinger anhänger darstellung thor
290,päpstliches bleisiegel bild heiligen petrus
291,leder gürtel langobardischer bronzeschließe
292,renaissance kupferbecken pflanzlichen mustern ...


In [ ]:
#Tokenizzazione
tfidfv = TfidfVectorizer()
X = tfidfv.fit_transform(corpus)
print(tfidfv.get_feature_names_out()) #stampa l'array contenente le parole del vocabolario creato

['acciaio' 'affresco' 'african' 'africana' 'africano' 'afrikanische'
 'afrikanischer' 'age' 'alfabeto' 'alphabet' 'alten' 'altägyptischen'
 'amazonasvölker' 'amazonian' 'amazzonici' 'amber' 'ambra' 'american'
 'americane' 'amphora' 'amphore' 'ancient' 'anello' 'anfora' 'anhänger'
 'animal' 'animali' 'antico' 'antiker' 'antikes' 'argento' 'armatura'
 'armor' 'arpa' 'ascia' 'assira' 'assyrian' 'assyrischen' 'avorio' 'axe'
 'aztec' 'azteco' 'aztekische' 'bacchetta' 'bacile' 'bambino' 'banchetto'
 'bankettszenen' 'banquet' 'barocca' 'barocco' 'barocke' 'barockgemälde'
 'barockharfe' 'baroque' 'bas' 'basin' 'bassorilievo' 'bastone'
 'battaglia' 'battle' 'battuto' 'bau' 'bc' 'belt' 'bergkristall'
 'bernstein' 'bestickter' 'beweglichem' 'bianchi' 'bianco' 'biblical'
 'bibliche' 'biblischen' 'bild' 'bizantino' 'black' 'blade' 'blau'
 'blauen' 'bleisiegel' 'blow' 'blu' 'blue' 'blumenmustern' 'bone'
 'borchie' 'bottiglia' 'bottle' 'bracciale' 'bracelet' 'brass'
 'breastplate' 'brocca' 'bronze' '

In [ ]:
print(X[0].toarray().shape)
print(X[0].toarray()) #stampa il vettore numerico che rappresenta la prima frase

(1, 810)
[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.     

In [ ]:
#Addestramento del modello di classificazione. Uso un MultinomialNB
y = df["Codice Lingua"].map({"it":1, "en":2, "de":3})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

In [ ]:
#Valutazione modello
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

#Il MultinomialNB ha raggiunto un'accuracy del 93%, con alte precisioni e recall su tutte e tre le lingue.
#Questo indica che riesce a distinguere molto bene italiano, inglese e tedesco basandosi sulle caratteristiche testuali.

              precision    recall  f1-score   support

           1       0.97      0.97      0.97        29
           2       0.90      0.93      0.92        30
           3       0.93      0.90      0.92        30

    accuracy                           0.93        89
   macro avg       0.93      0.93      0.93        89
weighted avg       0.93      0.93      0.93        89



In [ ]:
#Esempio di predizione della lingua su una nuova frase
nuova_frase = "Ho tra le mani un manoscritto medievale del VI secolo a.C."
X_nuovo = tfidfv.transform([text_cleaner(nuova_frase)])
pred = clf.predict(X_nuovo)
reverse_map = {1: "it", 2: "en", 3: "de"}
print(f"\n Predizione lingua per: \"{nuova_frase}\" → {reverse_map[pred[0]]}")


 Predizione lingua per: "Ho tra le mani un manoscritto medievale del VI secolo a.C." → it


In [ ]:
#Predizioni
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

#Probabilità per log_loss
y_train_proba = clf.predict_proba(X_train)
y_test_proba = clf.predict_proba(X_test)

#Accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

#Log loss
train_loss = log_loss(y_train, y_train_proba)
test_loss = log_loss(y_test, y_test_proba)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)
print("Train Log Loss:", train_loss)
print("Test Log Loss:", test_loss)

Train Accuracy: 1.0
Test Accuracy: 0.9325842696629213
Train Log Loss: 0.4832223453959423
Test Log Loss: 0.8377594105143642


In [ ]:
#Test con altro modello
from sklearn.linear_model import SGDClassifier
sgdc = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42)
sgdc.fit(X_train, y_train)

SGDClassifier(loss='log_loss', random_state=42)

In [ ]:
#Predizioni
y_train_pred = sgdc.predict(X_train)
y_test_pred = sgdc.predict(X_test)

#Probabilità predette (per log_loss)
y_train_proba = sgdc.predict_proba(X_train)
y_test_proba = sgdc.predict_proba(X_test)

#Accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

#Log loss
train_loss = log_loss(y_train, y_train_proba)
test_loss = log_loss(y_test, y_test_proba)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)
print("Train Log Loss:", train_loss)
print("Test Log Loss:", test_loss)

Train Accuracy: 1.0
Test Accuracy: 0.9325842696629213
Train Log Loss: 0.054068150069172094
Test Log Loss: 0.4574002411983212
